#### Building a Recommendation Engine with Association Rules
**Boréal Marché - from 1,000,000 baskets to a live application, using Apriori**
 

In [1]:
# Imports and environment check
import sys
import time
import pandas as pd
import numpy as np
from scipy import sparse

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

import gc
import tracemalloc

from pathlib import Path
import json
from datetime import datetime

print("Python executable:", sys.executable)
print("Python version:", sys.version)

Python executable: c:\Users\KELLY DESKTOP\Desktop\AI\Vanier\Unsupervised Machine Learning\Assignment\boreal-reco-Kelly_Baron\.venv\Scripts\python.exe
Python version: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


## Phase 1: Load and prepare

### 1.1 Products catalogue
The products catalogue is a CSV file with the products available in the store.
It is located in the `data` folder and is named `products.csv`.

In [2]:
file_path = '../data/products.csv'
products_df = pd.read_csv(file_path)

print("Products catalog loaded successfully.")
print("Number of products:", len(products_df))
print("Columns in the products catalog:", products_df.columns.tolist())
print("First 5 rows of the products catalog:")
print(products_df.head())

Products catalog loaded successfully.
Number of products: 98
Columns in the products catalog: ['Item', 'Item_EN', 'Category', 'Price']
First 5 rows of the products catalog:
                 Item         Item_EN  Category  Price
0      Steak de boeuf      Beef steak   Viandes  18.99
1           Saucisses        Sausages   Viandes   7.49
2  Poitrine de poulet  Chicken breast   Viandes  12.99
3               Bacon           Bacon   Viandes   6.99
4        Saumon frais    Fresh salmon  Poissons  16.49


### 1.2 Catalogue sanity checks
1. Shape and column names
2. Data types of all columns
3. Missing values per column
4. `Item` is unique, since it is the primary key of the catalogue
5. Categories and number of products per category
6. Price range, to spot unusually expensive products

In [3]:
# check 1: Shape and column names
print("Shape of the products catalog:", products_df.shape)
print("Columns in the products catalog:", products_df.columns.tolist())

# check 2: Data types of all columns
print("Data types of all columns:")
print(products_df.dtypes)

# check 3: Missing values per column
print("Missing values per column:")
print(products_df.isnull().sum())

# check 4: Item is unique
print("Is Item unique?", products_df['Item'].is_unique)

# check 5: Categories and number of products per category
print("Categories and number of products per category:")
print(products_df['Category'].value_counts())

# check 6: Price range
print("Price range:")
print(products_df['Price'].min(), "to", products_df['Price'].max())

# Additional check: Item_EN
print("Additional checks for Item_EN:")
print("Is Item_EN unique?", products_df['Item_EN'].is_unique)

Shape of the products catalog: (98, 4)
Columns in the products catalog: ['Item', 'Item_EN', 'Category', 'Price']
Data types of all columns:
Item            str
Item_EN         str
Category        str
Price       float64
dtype: object
Missing values per column:
Item        0
Item_EN     0
Category    0
Price       0
dtype: int64
Is Item unique? True
Categories and number of products per category:
Category
Épicerie             23
Fruits et légumes    13
Boissons              9
Collations            8
Boulangerie           7
Produits laitiers     6
Maison                6
Viandes               5
Fromages              5
Spécialités           5
Surgelés              4
Bébé                  4
Poissons              3
Name: count, dtype: int64
Price range:
1.29 to 89.99
Additional checks for Item_EN:
Is Item_EN unique? True


### 1.3 Load the transactions
The transactions file is in long format: one row per item, not one row per basket.
Only `TransactionID` and `Item` are loaded, because association rules only need to know
which items were in the same basket. `Item` is stored as a category (98 distinct names)
and `TransactionID` as a 32-bit integer to keep memory low.

In [4]:
# Load and time
transactions_path = '../data/transactionsdata.csv'
columns_to_load = ["TransactionID", "Item"]
column_types = {"TransactionID": "int32", "Item": "category"}

start_time = time.perf_counter()

transactions = pd.read_csv(transactions_path, usecols=columns_to_load, dtype=column_types)

load_seconds = time.perf_counter() - start_time

print(f"Load time: {load_seconds:.1f} seconds")

Load time: 0.9 seconds


In [5]:
# Inspection of transactions data memory usage and shape
memory_bytes = transactions.memory_usage(deep=True).sum()
memory_megabytes = memory_bytes / (1024 ** 2)

print(f"Memory usage: {memory_megabytes:.1f} MB")
print(f"Shape: {transactions.shape}")
print(transactions.dtypes)

Memory usage: 40.0 MB
Shape: (8379837, 2)
TransactionID       int32
Item             category
dtype: object


### 1.4 Transactions sanity checks
Before building the boolean table, I verify that the transactions match the
description in the brief and that they are consistent with the products catalogue.

1. Number of unique baskets and unique items
2. Basket sizes: mean, minimum, maximum, and number of single-item baskets
3. `TransactionID` range, and whether the IDs are contiguous
4. Catalogue match in both directions: items in the transactions that are missing from
   the catalogue, and catalogue products that never appear in the transactions
5. Duplicate items within a basket, and missing values

In [6]:
# Counts, basket sizes and ID range
# Check 1: unique baskets and unique items
unique_baskets = transactions['TransactionID'].nunique()
unique_items = transactions['Item'].nunique()

print(f"    number of unique baskets: {unique_baskets:,}")
print(f"    number of unique items: {unique_items}")

# Check 2: basket sizes
basket_sizes = transactions.groupby('TransactionID').size()

print(f"    average basket size: {basket_sizes.mean():.2f} items")
print(f"    smallest basket: {basket_sizes.min()} items")
print(f"    largest basket: {basket_sizes.max()} items")
print(f"    baskets with a single item: {(basket_sizes == 1).sum():,}")

# Check 3: TransactionID range
min_id = transactions['TransactionID'].min()
max_id = transactions['TransactionID'].max()
id_span = max_id - min_id + 1

print(f"    TransactionID range: {min_id} to {max_id}")
print(f"    IDs are contiguous: {id_span == unique_baskets}")

    number of unique baskets: 1,000,000
    number of unique items: 98
    average basket size: 8.38 items
    smallest basket: 1 items
    largest basket: 18 items
    baskets with a single item: 1,359
    TransactionID range: 100000 to 1099999
    IDs are contiguous: True


**Result:** the file contains 1,000,000 baskets and 98 distinct items, as the brief
states. The average basket has 8.38 items, ranging from 1 to 18. 1,359 baskets contain
a single item: they cannot support a rule between two items, but they still count towards
each item's support and the total number of baskets. `TransactionID` runs from 100000 to
1099999 with no gaps, so a basket's row position in the boolean table is its ID minus
100000.

In [7]:
# Catalogue match, both directions
# Check 4: catalogue match, using the original names on both sides
catalogue_items = set(products_df['Item'])
transaction_items = set(transactions['Item'].unique())

missing_from_catalogue = transaction_items - catalogue_items
never_purchased = catalogue_items - transaction_items

print(f"    items in the transactions but not in the catalogue: {len(missing_from_catalogue)}")
print(f"    catalogue products never purchased: {len(never_purchased)}")

    items in the transactions but not in the catalogue: 0
    catalogue products never purchased: 0


**Result:** every item in the transactions exists in the catalogue, and every catalogue
product was purchased at least once (0 differences in both directions). The 98 columns
of the boolean table will therefore match the 98 catalogue products one to one, and
every recommendation can be priced and categorised in the app.

In [8]:
# Duplicates and missing values
# Check 5: duplicates inside a basket, and missing values (reported separately)
duplicated_pairs = transactions.duplicated(subset=['TransactionID', 'Item']).sum()
missing_values = transactions.isnull().sum().sum()

print(f"    same item twice in the same basket: {duplicated_pairs:,}")
print(f"    missing values in the transactions: {missing_values:,}")

    same item twice in the same basket: 0
    missing values in the transactions: 0


**Result:** no item appears twice in the same basket and there are no missing values.
Converting to a boolean table will not lose any information, and the number of rows
in the long format (8,379,837) equals the number of ticks in the wide table.

#### Corrections made during these checks
My first version of the checks gave three results that contradicted the brief. I traced
each one to its cause:

- **876,967 "unique baskets".** I had counted distinct basket *contents*: many baskets
  contain exactly the same items, so they collapsed together. The correct count uses
  distinct `TransactionID` values: 1,000,000.
- **98 items missing from the catalogue.** I had converted the transaction names to
  lowercase but compared them with the capitalised catalogue names, so every name looked
  different. Lowercasing also converted the `Item` column from category to plain text,
  which undid the memory saving. I removed that step: the original names already match
  the catalogue exactly.
- **8,379,739 duplicated rows.** I had checked repeated *item names* across the whole
  file, which is expected since every product is bought many times. The correct check is
  a repeated (`TransactionID`, `Item`) pair, which gives 0.

### 1.5 Long to wide: building the boolean table
Apriori needs a boolean table with one row per basket and one column per product.
The transactions are in long format (one row per item), so they must be converted.
A naive pivot can exhaust memory or take several minutes at this scale, so I measure
the approach I chose.

**Approach B: integer codes and a sparse matrix**
1. Each purchase already has an integer code for its product, from the `Item` category.
2. Each basket's row number is its `TransactionID` minus 100000 (the IDs are contiguous).
3. A sparse matrix of shape (1,000,000, 98) is built from the two arrays, then converted
   to a boolean DataFrame with the 98 product names as column labels.

**Checks on the result**
1. Shape is (1,000,000, 98) and the dtype is boolean
2. The total number of True cells equals the number of rows in the long format (8,379,837)
3. Each column's sum equals that product's row count in the transactions
4. Build time and memory of the final table

In [9]:
# 1.5 Approach B: interger codes and sparse matrix
start_time = time.perf_counter()

# Column number of each purchase: the integer code of the item in the catalogue (conversion from category to integer code)
column_numbers = transactions['Item'].cat.codes.to_numpy()

# Product names, kept in the same order as the integer codes, for later reference
product_names = transactions['Item'].cat.categories.tolist()

# Row number of each purchase: basket ID minus the minimum basket ID (to start from 0)
first_id = transactions['TransactionID'].min()
row_numbers = (transactions['TransactionID'] - first_id).to_numpy()

# shape of the table
number_of_baskets = unique_baskets
number_of_products = len(product_names)

# Sanity check of the two arrays: column_numbers and row_numbers should have the same length as the number of transactions
print("Purchases:", len(row_numbers))
print("Row numbers range:", row_numbers.min(), "to", row_numbers.max())
print("Column numbers range:", column_numbers.min(), "to", column_numbers.max())
print("Column number dtype:", column_numbers.dtype)

# One True value per purchase
values = np.ones(len(row_numbers), dtype=bool)

# Sparse matrix: one row per basket, one column per product. 
sparse_basket = sparse.coo_matrix(
    (values, (row_numbers, column_numbers)),
    shape=(number_of_baskets, number_of_products)
)

# Dense boolean table with product names as column labels
dense_basket = sparse_basket.toarray()
basket =  pd.DataFrame(dense_basket, columns=product_names)

build_seconds = time.perf_counter() - start_time

# Memory usage of the final table
basket_memory_bytes = basket.memory_usage(deep=True).sum()
basket_memory_megabytes = basket_memory_bytes / (1024 ** 2)

print(f"Build time: {build_seconds:.2f} seconds")
print(f"Memory usage of the basket table: {basket_memory_megabytes:.1f} MB")

Purchases: 8379837
Row numbers range: 0 to 999999
Column numbers range: 0 to 97
Column number dtype: int8
Build time: 0.56 seconds
Memory usage of the basket table: 93.5 MB


In [10]:
# Check 1: shape and data types of the basket table
print("Shape of the basket table:", basket.shape)
print("Data types of the basket table:")
print(basket.dtypes.unique())

# Check 2: total number of True cells equal to the rows of the long format
total_ticks = int(basket.sum().sum())
print(f"Total True cells: {total_ticks:,}")
print("Matches the long format number of transactions:", total_ticks == len(transactions))

# check 3: each column sum equals that product's number of purchases in the long format
item_counts = transactions['Item'].value_counts()
item_counts = item_counts.reindex(product_names)
column_sums = basket.sum()
print("Column sums match product counts:", (column_sums.to_numpy() == item_counts.to_numpy()).all())

# check 4: column names match the catalogue
print("Column match the catalogue:", set(basket.columns) == set(products_df['Item']))

Shape of the basket table: (1000000, 98)
Data types of the basket table:
[dtype('bool')]
Total True cells: 8,379,837
Matches the long format number of transactions: True
Column sums match product counts: True
Column match the catalogue: True


In [11]:
# Saving the basket table to a Parquet file
basket.to_parquet('../data/basket.parquet')

**Result:** the boolean table has shape 1,000,000 × 98 and dtype `bool`. It was built in
0.59 seconds (including the integer coding of baskets and products) and occupies
93.5 MB. The total number of True cells is 8,379,837, equal to the number of rows in the
long format, every column sum matches its product's count in the transactions, and the
column names match the catalogue, so no purchase was lost or duplicated in the conversion.

### 1.6 Comparison with the encoder-based approach
Approach A builds a list of items for each basket and encodes it with `TransactionEncoder`.
I test it on a sample of 50,000 baskets and extrapolate to the full data, instead of
running it on all 1,000,000 baskets.

In [12]:
# Approach A on a sample of 50,000 baskets
sample_mask = transactions['TransactionID'] < 150000
sample = transactions[sample_mask]

start_time = time.perf_counter()

sample_baskets = sample.groupby('TransactionID', observed=True)['Item'].apply(list).tolist()
te = TransactionEncoder()
translated_sample = te.fit(sample_baskets).transform(sample_baskets)
sample_table = pd.DataFrame(translated_sample, columns=te.columns_)

sample_seconds = time.perf_counter() - start_time

print(f"Sample baskets: {len(sample_baskets):,}")
print(f"Time on sample: {sample_seconds:1f} seconds")
print(f"Estimated time for 1,000,000 baskets: {sample_seconds * 20:.0f} seconds")

Sample baskets: 50,000
Time on sample: 4.005974 seconds
Estimated time for 1,000,000 baskets: 80 seconds


**Result:** the sample of 50,000 baskets took 3.9 seconds, so the full data would take about
78 seconds, assuming the cost scales linearly (a rough estimate, since it was not run at full
size), against 0.59 seconds for Approach B, about 130 times faster. **I chose Approach B**
because it never builds Python lists of baskets: it writes the 8,379,837 (basket, product)
pairs directly into the table, and the final table (93.5 MB) fits easily in memory.

### 1.7 Product frequencies
`min_support` is a share of baskets, so I measure how many baskets contain each product.
I show the 10 most frequent and the 5 rarest products, as counts and as shares.

In [13]:
# How many baskets contain each product
product_counts = basket.sum()
product_counts = product_counts.sort_values(ascending=False)

# Prodcut shares of all baskets
product_shares = product_counts / len(basket)

print("10 most frequent products:")
print(product_shares.head(10).round(4))

print("5 rarest products:")
print(product_counts.tail(5))
print(product_shares.tail(5).round(5))

10 most frequent products:
Sac réutilisable     0.9000
Biscuits             0.1614
Noix mélangées       0.1552
Beurre d'arachide    0.1537
Bière                0.1505
Savon à vaisselle    0.1498
Fromage cheddar      0.1435
Charbon de bois      0.1406
Papier hygiénique    0.1398
Céréales             0.1392
dtype: float64
5 rarest products:
Safran                         818
Huile de truffe                817
Caviar                         810
Vinaigre balsamique vieilli    779
Foie gras                      756
dtype: int64
Safran                         0.00082
Huile de truffe                0.00082
Caviar                         0.00081
Vinaigre balsamique vieilli    0.00078
Foie gras                      0.00076
dtype: float64


**Result:** the most frequent product is Sac réutilisable (reusable bag), present in 90.0% of
baskets, far ahead of the next one (Biscuits, 16.1%). The 5 rarest products (Foie gras,
Vinaigre balsamique vieilli, Caviar, Huile de truffe and Safran) appear in about 0.08% of
baskets each (756 to 818 baskets). That is below a `min_support` of 0.005 (5,000 baskets), so
they cannot appear in any rule at the thresholds I plan to test.

### 1.8 Prediction before mining
**Written before running Apriori.**

- **Product that will cause trouble:** `Sac réutilisable`, present in 90% of baskets.
- **Why:** because it is in almost every basket, the confidence of any rule ending in it will be
  close to 90%, whatever the other item is. The store would learn nothing from customers who buy X also buy a reusable bag, because customers would have bought the bag anyway.
- **Which ranking it will distort:** the ranking by confidence. Rules ending in
  `Sac réutilisable` will have a confidence of about 90%, the highest available for almost any
  item, so I expect them to fill most of the top 10 by confidence. Their lift will be
  about 1 (90% / 90%), meaning no more often than average, so in the ranking by lift
  they will drop out of the top. The top of the lift ranking should instead
  show pairs bought together for the same meal.
- **Products I expect to be missing from the rules:** the five rarest products (Foie gras,
  Vinaigre balsamique vieilli, Caviar, Huile de truffe, Safran), because each appears in
  only 0.076% to 0.082% of baskets, below every threshold I plan to test (0.05, 0.02,
  0.01, 0.005). They never reach the minimum support, so no rule can contain them.

### 1.9 Possible duplicate products
The only pair of products in the catalogue with almost identical names is `Baguette at 2.99$` and `Pain baguette rustique at 3.99$`. If they were one product recorded twice, customers would rarely buy in the same basket.

In [14]:
baguette = basket['Baguette']
baguette_rustic = basket['Pain baguette rustique']

only_baguette = (baguette & ~baguette_rustic).sum()
only_rustic = (baguette_rustic & ~baguette).sum()
both = (baguette & baguette_rustic).sum()
expected_both = baguette.mean() * baguette_rustic.mean() * len(basket)

print("Baguette total:", baguette.sum())
print("Rustic bagette total:", baguette_rustic.sum())
print("Only baguette:", only_baguette)
print("Only rustic baguette:", only_rustic)
print("Both in the same basket:", both)
print("Expected together if independant:", round(expected_both))

Baguette total: 69400
Rustic bagette total: 42219
Only baguette: 66391
Only rustic baguette: 39210
Both in the same basket: 3009
Expected together if independant: 2930


**Result:** 69,400 baskets contain Baguette and 42,219 contain the rustic baguette. 3,009
contain both, against about 2,930 expected if the two were independent. They are bought
independently rather than as substitutes. **Decision: do not merge.** The prices differ, the
purchases are independent, and merging would hide a seasonal product that Phase 4 needs. I
also no other pair of products in the catalogue has near-identical names that almost never share a basket and
found none, so no other pair is a better candidate.

**Update**: Test from the cell 3.3 finding the Top 10 pairs by lift indicate that Rustic baguette has high lift with Fromage à raclette. Not Pain baguette that was not listed. It means that both baguette are completly two different product and should be treated seperatly.

### Phase 1 summary
The data is loaded and verified (1,000,000 baskets, 98 products, no duplicates or gaps), the
boolean table is built in 0.59 seconds and 93.5 MB, and my prediction is written before any
mining. Next: Apriori.

## Phase 2: Mine the rules with Apriori
I use `mlxtend.frequent_patterns.apriori` on the boolean table (1,000,000 × 98) at four
values of `min_support` (0.05, 0.02, 0.01, 0.005). For each run I record the number of
itemsets, the number of rules, the time and the peak memory, and I document what happens
with the default settings.

### 2.1 Functions for one run
`run_apriori` times one call of `apriori` and measures its peak memory with `tracemalloc`; it catches
a `MemoryError` and returns it as the outcome. `record_run` stores the itemsets, saves them to disk and
appends the run to `reports/apriori_runs.csv`, so a crash cannot erase earlier results.

In [15]:
run_log = []
apriori_results = {}

def run_apriori(basket_table, min_support, low_memory):
    gc.collect()
    tracemalloc.start()
    start_time = time.perf_counter()

    try:
        itemsets = apriori(
            basket_table,
            min_support=min_support,
            use_colnames=True,
            low_memory=low_memory
        )
        outcome = "success"
    except MemoryError:
        itemsets = None
        outcome = 'MemoryError'

    seconds = time.perf_counter() - start_time
    current_bytes, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    peak_megabytes = peak_bytes / (1024 ** 2)

    return itemsets, seconds, peak_megabytes, outcome


def record_run(basket_table, min_support, low_memory):
    itemsets, seconds, peak_mb, outcome = run_apriori(basket_table, min_support, low_memory)

    if itemsets is not None:
        apriori_results[(min_support, low_memory)] = itemsets
        itemsets.to_pickle(f"../data/itemsets_{min_support}_{low_memory}.pkl")
        number_of_itemsets = len(itemsets)
        largest_itemset = int(itemsets['itemsets'].apply(len).max())
    else:
        number_of_itemsets = None
        largest_itemset = None

    run_log.append({
        "min_support": min_support,
        "low_memory": low_memory,
        "itemsets": number_of_itemsets,
        "largest_itemset": largest_itemset,
        "seconds": round(seconds, 2),
        "peak_memory_mb": round(peak_mb, 1),
        "outcome": outcome,
    })
    pd.DataFrame(run_log).to_csv("../reports/apriori_runs.csv", index=False)
    print(run_log[-1])

### 2.2 Sample test at 0.05 and scaling check

In [16]:
sample_basket = basket.sample(10000, random_state=42)

itemsets, seconds, peak_mb, outcome = run_apriori(sample_basket, 0.05, False)
print(f"default: {len(itemsets)} itemsets, {seconds:.2f} s, {peak_mb:.1f} MB, {outcome}")

itemsets, seconds, peak_mb, outcome = run_apriori(sample_basket, 0.05, True)
print(f"low_memory=True: {len(itemsets)} itemsets, {seconds:.2f} s, {peak_mb:.1f} MB, {outcome}")

default: 195 itemsets, 0.06 s, 75.3 MB, success
low_memory=True: 195 itemsets, 0.04 s, 1.1 MB, success


**Result:** on 10,000 baskets at 0.05, both settings found the same 195 itemsets. The default
peaked at 75.3 MB and `low_memory=True` at 1.1 MB, about 70 times less. The parameter changes how
the work is done, not the answer.

In [17]:
for size in [10000, 50000, 100000]:
    sample_basket = basket.sample(size, random_state=42)
    itemsets, seconds, peak_mb, outcome = run_apriori(sample_basket, 0.05, False)
    print(f"{size:,} baskets: {peak_mb:.1f} MB, {seconds:.2f} s, {outcome}")

10,000 baskets: 75.3 MB, 0.06 s, success
50,000 baskets: 366.1 MB, 0.27 s, success
100,000 baskets: 752.7 MB, 0.61 s, success


**Result:** the default setting's peak memory grows linearly with the number of baskets (75.3 MB,
366.1 MB and 752.7 MB for 10,000, 50,000 and 100,000), about 7.5 MB per 1,000 baskets. At
1,000,000 baskets it would need about 7.5 GB at 0.05.

### 2.3 Default setting at lower thresholds: sample test and projection

In [18]:
sample_basket = basket.sample(10000, random_state=42)

for min_support in [0.02, 0.01, 0.005]:
    itemsets, seconds, peak_mb, outcome = run_apriori(sample_basket, min_support, False)
    print(f"default, {min_support}: {len(itemsets)} itemsets, {seconds:.2f} s, {peak_mb:.1f} MB")

default, 0.02: 659 itemsets, 0.12 s, 122.5 MB
default, 0.01: 2327 itemsets, 0.32 s, 232.8 MB
default, 0.005: 7231 itemsets, 1.16 s, 1018.4 MB


**Result:** on 10,000 baskets the default setting peaks at 122.5 MB at 0.02, 232.8 MB at 0.01 and
1,018.4 MB at 0.005. Projected to 1,000,000 baskets (memory grew linearly with the number of
baskets in the scaling check), that is about 12.3 GB, 23.3 GB and 101.8 GB, against 16 GB of
RAM. The number of itemsets found on the sample rises from 659 to 2,327 to 7,231 as the threshold
falls, so more candidates have to be tested at once. I therefore expect the default setting to fail
from 0.01 downward, and to be borderline at 0.02. These are projections from a sample, not
full-table measurements.

### 2.4 Full runs with `low_memory=True`
Each threshold is run once on the full table (1,000,000 baskets). `record_run` prints the result and
appends it to `reports/apriori_runs.csv`. The itemsets are saved in `data/` for the next phases.

In [19]:
record_run(basket, 0.05, True)

{'min_support': 0.05, 'low_memory': True, 'itemsets': 191, 'largest_itemset': 4, 'seconds': 2.44, 'peak_memory_mb': 106.5, 'outcome': 'success'}


In [20]:
record_run(basket, 0.02, True)

{'min_support': 0.02, 'low_memory': True, 'itemsets': 658, 'largest_itemset': 5, 'seconds': 6.05, 'peak_memory_mb': 109.9, 'outcome': 'success'}


In [21]:
record_run(basket, 0.01, True)

{'min_support': 0.01, 'low_memory': True, 'itemsets': 2267, 'largest_itemset': 7, 'seconds': 14.44, 'peak_memory_mb': 109.9, 'outcome': 'success'}


In [22]:
record_run(basket, 0.005, True)

{'min_support': 0.005, 'low_memory': True, 'itemsets': 7106, 'largest_itemset': 7, 'seconds': 32.09, 'peak_memory_mb': 110.0, 'outcome': 'success'}


In [23]:
runs_table = pd.read_csv("../reports/apriori_runs.csv")
print(runs_table.to_string(index=False))

 min_support  low_memory  itemsets  largest_itemset  seconds  peak_memory_mb outcome
       0.050        True       191                4     2.44           106.5 success
       0.020        True       658                5     6.05           109.9 success
       0.010        True      2267                7    14.44           109.9 success
       0.005        True      7106                7    32.09           110.0 success


**Result:** with `low_memory=True`, all four thresholds ran on the full table of 1,000,000 baskets.
Peak memory stayed between 106.5 and 110.0 MB at every threshold, close to the size of the
table itself (93.5 MB), so the search for candidates adds little. The time grew from 2.5 s at 0.05
to 32.0 s at 0.005, the number of itemsets from 191 to 7,106, and the largest itemset from 4
to 7 products. Compared with the projected peak of the default setting (about 7.5 GB at 0.05 and
101.8 GB at 0.005), `low_memory=True` needs about 70 and 900 times less memory. Times include the
overhead of `tracemalloc`.

### 2.5 Rules for each threshold

In [24]:
confidence_threshold = 0.3 # A value of 0.3 means at least 3 in 10 baskets that contain X also contain Y
rules_by_support = {}
rules_summary = []

for min_support in [0.05, 0.02, 0.01, 0.005]:
    itemsets = apriori_results[(min_support, True)]

    start_time = time.perf_counter()
    rules = association_rules(
        itemsets,
        num_itemsets=len(basket),
        metric='confidence',
        min_threshold=confidence_threshold
    )
    rules_seconds = time.perf_counter() - start_time

    rules_by_support[min_support] = rules
    rules_summary.append({
        "min_support": min_support,
        "itemsets": len(itemsets),
        "rules": len(rules),
        "rules_with_lift_above_1": int((rules['lift'] > 1).sum()),
        "rules_seconds": round(rules_seconds, 2)
    })

print(pd.DataFrame(rules_summary).to_string(index=False))

 min_support  itemsets  rules  rules_with_lift_above_1  rules_seconds
       0.050       191    282                      254           0.00
       0.020       658   2707                     2585           0.01
       0.010      2267   7817                     7386           0.03
       0.005      7106  23918                    22476           0.19


**Result:** the number of rules (confidence of at least 0.3) grows from 282 at 0.05 to 23,918 at
0.005, and 90% or more of them have lift above 1 (254 of 282, 22,476 of 23,918). Generating the
rules takes under 0.2 s, so the cost of the pipeline is in the Apriori step.

### 2.6 Candidate rules

### 2.6 Rules for the model: choice of thresholds
The model uses the itemsets of the lowest threshold I tested (`min_support` = 0.005), the most
complete set. The minimum confidence decides how many products can ever be recommended, so I
compare several values before choosing one.

In [25]:
# The candidate rule set (min_confidence 0.05)
model_min_support = 0.005
model_min_confidence = 0.05

model_itemsets = apriori_results[(model_min_support, True)]

model_rules = association_rules(
    model_itemsets,
    num_itemsets=len(basket),
    min_threshold=model_min_confidence
)

model_rules.to_pickle('../model_rules_candidate.pkl')

print(f"Rules: {len(model_rules):,}")
print(f"Rules with lift above 1: {(model_rules['lift'] > 1).sum():,}")

Rules: 57,063
Rules with lift above 1: 47,393


**Correction.** This count treats a product as recommendable if it appears anywhere in a consequent,
including rules with several products on the right-hand side, where a product can "ride along" with
another that has the real link. Counting only rules with a single-product consequent (what the app
uses) and the lift > 1.2 filter gives 57 of 98 products at every confidence level (9,256 rules at 0.05,
9,077 at 0.3). I keep `min_confidence = 0.05`, since the confidence threshold hardly changes the rule set
once lift filters out the noise.

In [26]:
# The evidence for the confidence threshold

for confidence in [0.05, 0.1, 0.2, 0.3]:
    candidate_rules = association_rules(
        model_itemsets,
        num_itemsets=len(basket),
        metric='confidence',
        min_threshold=confidence
    )
    useful_rules = candidate_rules[candidate_rules['lift'] > 1]

    covered_products = set()
    for consequent in useful_rules['consequents']:
        covered_products = covered_products | set(consequent)

    print(f"min_confidence {confidence}: {len(useful_rules):,} rules, "
          f"{len(covered_products)} products can be recommended")

min_confidence 0.05: 47,393 rules, 92 products can be recommended
min_confidence 0.1: 36,475 rules, 74 products can be recommended
min_confidence 0.2: 26,991 rules, 58 products can be recommended
min_confidence 0.3: 22,476 rules, 58 products can be recommended


**Choice of thresholds.**

- **`min_support = 0.005`.** It is the lowest threshold I tested and ran in 32 s with
  `low_memory=True`. 93 of the 98 products are frequent enough at this level, and the 5 rarest
  (about 0.08% of baskets each) can never appear in a rule.
- **`min_confidence = 0.05`.** The number of products that can ever be recommended depends on it
  (rules with lift above 1):

| min confidence | Rules | Products that can be recommended |
|---|---|---|
| 0.05 | 47,393 | 92 |
| 0.1 | 36,475 | 74 |
| 0.2 | 26,991 | 58 |
| 0.3 | 22,476 | 58 |

At 0.3, 40 of the 98 products could never be recommended, which is the reinforcement bias the brief
warns about: only products already popular would be suggested. A low confidence is acceptable here
because the app ranks suggestions by lift and shows the confidence. For example, a product bought
with X in 5% of baskets, against 1% of baskets in general (a lift of 5), still ranks above a
popular product with no real link to X (a lift of about 1).

## Phase 3: Evaluate the rules
All the rules below come from `model_rules` (`min_support` 0.005, `min_confidence` 0.05).

- **Support:** the share of all baskets that contain the item or the combination of items.
- **Confidence:** among the baskets that contain X, the share that also contain Y.
- **Lift:** how many times more often Y appears in baskets with X than in baskets in general
  (1 means no link, above 1 means bought together more than chance, below 1 means less).

### 3.1 Simple rules (one item on each side):



In [27]:
antecedent_size = model_rules['antecedents'].apply(len)
consequent_size = model_rules['consequents'].apply(len)

simple_rules = model_rules[(antecedent_size == 1) & (consequent_size == 1)].copy() # Copy to prevent the accidental altering of the original data because of Pandas creating a set of pointers to 'View' the original model-rules DataFrame.
simple_rules['antecedent'] = simple_rules['antecedents'].apply(lambda items: next(iter(items))) # Convert the set or list of items into a iterator because frozensets do not support indexing
simple_rules['consequent'] = simple_rules['consequents'].apply(lambda items: next(iter(items)))

columns_to_show = ['antecedent', 'consequent', 'support', 'confidence', 'lift']
print(f"Simple rules: {len(simple_rules):,}")

Simple rules: 3,884


### 3.2 Top 10 by confidence

In [28]:
top_by_confidence = simple_rules.sort_values('confidence', ascending=False)
print(top_by_confidence[columns_to_show].head(10).round(3).to_string(index=False))

       antecedent       consequent  support  confidence  lift
        Concombre Sac réutilisable    0.063       0.903 1.003
   Cidre de glace Sac réutilisable    0.053       0.902 1.002
     Sauce tomate Sac réutilisable    0.096       0.902 1.002
           Relish Sac réutilisable    0.053       0.902 1.002
    Huile d'olive Sac réutilisable    0.049       0.902 1.002
           Pommes Sac réutilisable    0.082       0.902 1.002
Moutarde de Dijon Sac réutilisable    0.074       0.902 1.002
   Fromage suisse Sac réutilisable    0.084       0.901 1.002
   Steak de boeuf Sac réutilisable    0.095       0.901 1.002
           Avocat Sac réutilisable    0.040       0.901 1.001


**Top 10 by confidence.** All 10 rules end in the reusable bag, with a confidence of about 90%.
This does not mean that the antecedent leads customers to buy a bag: the bag is in about 90% of all
baskets, so any group of baskets, whatever it contains, shows about 90% of bags. The lift, between
1.001 and 1.003, confirms it: knowing that X is in the basket does not make the bag any more
likely, so the two products are independent and the rule gives the store no new information. A
plausible reason why the bag is so common is that customers need something to carry their
purchases, but the data only shows that it is common, not why. The antecedents are ordinary
products present in about 4% to 11% of baskets (for example Sauce tomate and Steak de boeuf), not
the most popular ones, and their order in the list reflects tiny variations in confidence (0.901
to 0.903), so the ranking itself carries no real information. This matches my prediction in 1.8:
ranking by confidence is fooled by a very common product.

### 3.3 Top 10 by lift

In [29]:
top_by_lift = simple_rules.sort_values('lift', ascending=False)
print(top_by_lift[columns_to_show].head(10).round(3).to_string(index=False))

            antecedent             consequent  support  confidence   lift
    Fromage à raclette      Bouillon à fondue    0.034       0.491 10.171
     Bouillon à fondue     Fromage à raclette    0.034       0.697 10.171
          Charcuteries     Fromage à raclette    0.032       0.683  9.976
    Fromage à raclette           Charcuteries    0.032       0.466  9.976
    Fromage à raclette             Cornichons    0.030       0.434  9.773
            Cornichons     Fromage à raclette    0.030       0.670  9.773
Pain baguette rustique     Fromage à raclette    0.028       0.654  9.552
    Fromage à raclette Pain baguette rustique    0.028       0.403  9.552
           Algues nori            Riz à sushi    0.054       0.795  9.505
           Riz à sushi            Algues nori    0.054       0.651  9.505


**Top 10 by lift.** The list is dominated by groups of products bought for the same meal: raclette
cheese, fondue broth, cold cuts (Charcuteries), pickles (Cornichons) and rustic baguette, plus sushi rice
and seaweed (Algues nori). Their confidence (0.40 to 0.80) is lower than the 0.90 of the first list, but
their lift is about 10: for example, fondue broth is in only about 4.8% of all baskets, yet in 49% of
the baskets with raclette cheese. Unlike the reusable bag, the high confidence does not come from the
product being popular, so these rules tell the store something it did not already know. They also
look like complementary ingredients of one recipe: seaweed leads to sushi rice with a confidence of
0.80, since they are the main ingredients of the same dish.

Every pair appears twice, with the same support and the same lift but a different confidence. The
data has no order of purchase (a basket is a set of items), so this is not about which product
customers think of first. Support and lift describe the pair as a whole, while confidence divides by how
common the first product is: raclette cheese is in about 6.9% of baskets and fondue broth in about
4.8%, so 49% of raclette baskets contain fondue broth, but 70% of fondue baskets contain raclette.

### 3.4 Reusable bag rules

In [30]:
bag_rules = simple_rules[simple_rules['consequent'] == 'Sac réutilisable']
print(f"Simple rules ending in the reusable bag: {len(bag_rules)}")
print(f"From the rules, how many have a lift above 1: {(bag_rules['lift'] > 1).sum()}")
print(f"Lift range: {bag_rules['lift'].min():.3f} to {bag_rules['lift'].max():.3f}")
print(f"Top 10 by confidence ending in the bag: {(top_by_confidence.head(10)['consequent'] == 'Sac réutilisable').sum()}")

Simple rules ending in the reusable bag: 92
From the rules, how many have a lift above 1: 47
Lift range: 0.997 to 1.003
Top 10 by confidence ending in the bag: 10


### 3.5 Distribution of lift

In [31]:
lift_bins = pd.cut(model_rules['lift'], bins=[0, 0.95, 1.0, 1.05, 1.2, 1.5, 3, 20])
print(lift_bins.value_counts().sort_index())

lift
(0.0, 0.95]       12
(0.95, 1.0]     9658
(1.0, 1.05]     8059
(1.05, 1.2]        0
(1.2, 1.5]         0
(1.5, 3.0]       125
(3.0, 20.0]    39209
Name: count, dtype: int64


### 3.6 Example with very common itemset


In [32]:
common_example_rule = simple_rules[(simple_rules['antecedent'] == 'Couches') & (simple_rules['consequent'] == 'Bière')]
print(common_example_rule[columns_to_show].round(3).to_string(index=False))

antecedent consequent  support  confidence  lift
   Couches      Bière    0.031       0.366  2.43


**Comparison.** The top 10 by confidence all end in `Sac réutilisable`, with a confidence of about
90.2% and a lift between 1.001 and 1.003. The top 10 by lift are groups of products bought for the
same meal (raclette, fondue and sushi ingredients), with a confidence of only 40.3% to 79.5% but a
lift between 9.505 and 10.171. Confidence rewards rules that end in a very common product, while lift
divides out how common that product is, so a high confidence does not imply a high lift: the bag has the
highest confidence and no real lift, while raclette and fondue products have a lower confidence and a
lift of about 10.

**Was my prediction correct?** Yes. The trouble product was the reusable bag (about 90% of baskets),
the ranking it distorted was the ranking by confidence (10 of the top 10 end in the bag), its lift
was about 1, and the top of the lift ranking shows pairs bought for the same meal. The five
rarest products are absent from all the final rules (see 3.8). What I did not anticipate is that the filter
`lift > 1` is not enough on its own: it keeps 47 of the 92 simple rules ending in the bag, with
a lift of 1.001 to 1.003 (see 3.5 and 3.7).


**What this means for the store.** A product in almost every basket, like the reusable bag, cannot reveal
a pattern: its lift is capped near 1 (at most about 1.1), so any rule ending in it tells the store
nothing new, however high its confidence. Moderately frequent products with a high lift, such as
raclette, fondue broth and sushi ingredients, show real links between products, because they are bought
together far more often than chance. That makes lift, not confidence, the right measure to rank
suggestions, and `min_support` the safeguard that keeps a high lift from resting on only a few
baskets. A high lift still shows association, not cause (see the diapers and beer rule).

### 3.7 Filter: lift cutoff
The brief asks to keep only rules with lift above 1. I compare it with a stricter cutoff of 1.2,
chosen from the distribution of lift in 3.5.

In [33]:
for lift_cutoff in [1.0, 1.2]:
    kept_rules = model_rules[model_rules['lift'] > lift_cutoff]

    covered_products = set()
    for consequent in kept_rules['consequents']:
        covered_products = covered_products | set(consequent)

    ends_in_bag = kept_rules['consequents'].apply(lambda items: items == frozenset({'Sac réutilisable'}))

    print(f"lift above {lift_cutoff}: {len(kept_rules):,} rules, "
          f"{len(covered_products)} products can be recommended, "
          f"{ends_in_bag.sum():,} rules end in the reusable bag")

lift above 1.0: 47,393 rules, 92 products can be recommended, 1,832 rules end in the reusable bag
lift above 1.2: 39,334 rules, 90 products can be recommended, 0 rules end in the reusable bag


### 3.8 Final rules and coverage
The final rule set keeps rules with lift above 1.2, mined at `min_support` 0.005 with
`min_confidence` 0.05. These are the rules that will be saved in Phase 5.

In [34]:
final_rules = model_rules[(model_rules['lift'] > 1.2) & (model_rules['consequents'].apply(len) == 1)].copy()

covered_products = set()
for consequent in final_rules['consequents']:
    covered_products = covered_products | set(consequent)

not_covered = sorted(set(basket.columns) - covered_products)

print(f"Final rules: {len(final_rules):,}")
print(f"Products that can be recommended: {len(covered_products)}")
print(f"Products that can never be recommended: {len(not_covered)}")

# Why each uncovered product is not recommended: its share of baskets and its best lift as a single consequent
single_rules = model_rules[model_rules['consequents'].apply(len) == 1].copy()
single_rules['consequent'] = single_rules['consequents'].apply(lambda items: next(iter(items)))

best_lift = single_rules.groupby('consequent')['lift'].max()
product_share = basket.mean()

uncovered_table = pd.DataFrame({
    'share_of_baskets': product_share[not_covered].round(4),
    'best_lift': best_lift.reindex(not_covered).round(3),
})
print(uncovered_table.to_string())

Final rules: 9,256
Products that can be recommended: 57
Products that can never be recommended: 41
                             share_of_baskets  best_lift
Bagels                                 0.0627      1.018
Baguette                               0.0694      1.020
Beurre                                 0.1342      1.034
Beurre d'arachide                      0.1537      1.025
Biscuits                               0.1614      1.035
Carottes                               0.0314        NaN
Caviar                                 0.0008        NaN
Chocolat noir                          0.0727      1.030
Cidre de glace                         0.0582      1.018
Confiture de fraises                   0.0554      1.022
Crevettes                              0.1188      1.025
Crème glacée                           0.0918      1.024
Céréales                               0.1392      1.026
Détergent à lessive                    0.0792      1.038
Eau pétillante                         0.0820 

### Phase 4: seasonal analysis

### 4.1 One date per basket

In [35]:
date_raw = pd.read_csv(
    '../data/transactionsdata.csv',
    usecols=['TransactionID', 'Date'],
    dtype={'TransactionID': 'int32', 'Date': 'category'}

)

basket_dates = date_raw.drop_duplicates('TransactionID')
basket_dates = basket_dates.set_index('TransactionID')
basket_dates = basket_dates.reindex(range(int(first_id), int(first_id) + number_of_baskets))

basket_months = pd.to_datetime(basket_dates['Date'].astype(str)).dt.month.to_numpy()

### 4.2 Summer and winter basket

In [36]:
summer_months = np.isin(basket_months, [6, 7, 8])
winter_months = np.isin(basket_months, [12, 1, 2])

summer_basket = basket[summer_months]
winter_basket = basket[winter_months]

print(f"Summer baskets Jun to Aug: {len(summer_basket):,}")
print(f"Winter baskets Dec to February: {len(winter_basket):,}")

Summer baskets Jun to Aug: 252,474
Winter baskets Dec to February: 246,965


### 4.3 Mine each season

In [37]:
def season_rules(season_basket):
    itemsets = apriori(season_basket, min_support=0.005, use_colnames=True, low_memory=True)
    rules = association_rules(
        itemsets,
        num_itemsets=len(season_basket),
        metric='confidence',
        min_threshold=0.05,
    )
    rules = rules[rules['lift'] > 1.2]

    antecedent_size = rules['antecedents'].apply(len)
    consequent_size = rules['consequents'].apply(len)
    rules = rules[(antecedent_size == 1) & (consequent_size == 1)].copy()

    rules['antecedent'] = rules['antecedents'].apply(lambda items: next(iter(items)))
    rules['consequent'] = rules['consequents'].apply(lambda items: next(iter(items)))
    rules = rules.set_index(['antecedent', 'consequent'])

    return itemsets, rules[['support', 'confidence', 'lift']]


summer_itemsets, summer_rules = season_rules(summer_basket)
winter_itemsets, winter_rules = season_rules(winter_basket)

print(f"Summer: {len(summer_itemsets):,} itemsets, {len(summer_rules)} simple rules")
print(f"Winter: {len(winter_itemsets):,} itemsets, {len(winter_rules)} simple rules")

Summer: 9,867 itemsets, 274 simple rules
Winter: 6,042 itemsets, 320 simple rules


### 4.4 Comparaison of the season rules

In [38]:
shared_rules = winter_rules.join(summer_rules, lsuffix='_winter', rsuffix='_summer', how='inner')
shared_rules['lift_difference'] = (shared_rules['lift_winter'] - shared_rules['lift_summer']).abs()
print(shared_rules.sort_values('lift_difference', ascending=False).head(10).round(3).to_string())

winter_only = winter_rules.index.difference(summer_rules.index)
summer_only = summer_rules.index.difference(winter_rules.index)
print(f"Rules only in winter: {len(winter_only)}, only in summer: {len(summer_only)}")
print(winter_rules.loc[winter_only].sort_values('lift', ascending=False).head(10).round(3))

                                   support_winter  confidence_winter  lift_winter  support_summer  confidence_summer  lift_summer  lift_difference
antecedent       consequent                                                                                                                       
Steak de boeuf   Charbon de bois            0.025              0.625       12.661           0.187              0.943        3.499            9.162
Charbon de bois  Steak de boeuf             0.025              0.509       12.661           0.187              0.695        3.499            9.162
Pain à hamburger Charbon de bois            0.024              0.606       12.266           0.177              0.940        3.488            8.779
Charbon de bois  Pain à hamburger           0.024              0.477       12.266           0.177              0.657        3.488            8.779
Ketchup          Charbon de bois            0.021              0.588       11.907           0.157              0.932  

C:\Users\KELLY DESKTOP\AppData\Local\Temp\ipykernel_15660\4035978599.py:1: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  shared_rules = winter_rules.join(summer_rules, lsuffix='_winter', rsuffix='_summer', how='inner')


### 4.5 A rule that disapears in  one season but is present in the other

In [39]:
product_a = 'Bouillon à fondue'
product_b = 'Charcuteries'

for season_name, season_basket in [('summer', summer_basket), ('winter', winter_basket)]:
    both_sharing = (season_basket[product_a] & season_basket[product_b]).mean()
    print(f"{season_name}: {product_a} + {product_b} in {both_sharing:.4f} of baskets")

summer: Bouillon à fondue + Charcuteries in 0.0046 of baskets
winter: Bouillon à fondue + Charcuteries in 0.0401 of baskets


**Result: a rule that exists in winter and disappears in summer.** Fondue broth (Bouillon à fondue) and
cold cuts (Charcuteries) appear together in 4.01% of winter baskets but only 0.46% of summer baskets,
about 9 times less. In winter the rule `Bouillon à fondue → Charcuteries` is in the rule set (support
0.040, confidence 0.495, lift 6.3). In summer the pair's support (0.0046) falls below
`min_support` (0.005), so no rule is mined: the rule disappears from the summer rule set, although a
small number of baskets (about 1,160) still contain both products. The pair belongs to a winter
meal: fondue and raclette are cold-weather dishes, which is consistent with the seasonal pattern
of the rustic baguette found in 1.9 (it sells much more in winter).

### Conclusion of the seasonal analysis

**How often should Boréal Marché re-mine its rules, and why?** Re-mining quaterly could become a transitional job of each major retail season. This ensures seasonal trends are fully captured.
**What happens if the website serves year-round rules in January?** Information could be diluted by the 6-8 months of winter meaning the transaction of the product during this time will not be capture. One way to solve the problem is to use hybrid apporach where fixed seasonal schedule before big retail season and use short rolling window every month at the beginning

## Phase 5: Save the rules
The final rules (single-product consequent, lift above 1.2) are saved in `models/rules.json`.
`models/metadata.json` records the thresholds, the mining date, the number of rules and the coverage,
plus the list of products that can never be recommended and the most popular products, which the
app uses for its fallback and its explanations. The app loads these files and never runs Apriori.

In [40]:
Path('../models').mkdir(exist_ok=True)

rules_for_json = []
for _, rule in final_rules.sort_values('lift', ascending=False).iterrows():
    rules_for_json.append({
        "antecedent": sorted(rule['antecedents']),
        "consequent": next(iter(rule['consequents'])),
        "support": round(float(rule['support']), 6),
        "confidence": round(float(rule['confidence']), 6),
        "lift": round(float(rule['lift']), 6),
    })

with open('../models/rules.json', 'w', encoding='utf-8') as rules_file:
    json.dump(rules_for_json, rules_file, ensure_ascii=False, indent=1)

print(f"Rules saved: {len(rules_for_json):,}")

Rules saved: 9,256


In [43]:
min_lift = 1.2

not_recommendable = {}
for product in not_covered:
    share = float(product_share[product])
    best = best_lift.get(product)
    if share < model_min_support:
        reason = "below min_support: too rare to reach the threshold"
    else:
        reason = "frequent but bought independently of other products: no rule with lift above the cutoff"
    not_recommendable[product] = {
        "share_of_baskets": round(share, 6),
        "best_lift": None if best is None else round(float(best), 3),
        "reason": reason,
    }

popular_products = []
for product, share in product_share.sort_values(ascending=False).head(10).items():
    popular_products.append({"product": product, "share_of_baskets": round(float(share), 6)})

mining_run = [row for row in run_log if row['min_support'] == model_min_support and row['low_memory']][-1]

metadata = {
    "mining_date": datetime.today().isoformat(),
    "algorithm": "mlxtend apriori (low_memory=True) + association_rules",
    "data": {"baskets": int(len(basket)), "products": int(basket.shape[1])},
    "thresholds": {
        "min_support": model_min_support,
        "min_confidence": model_min_confidence,
        "min_lift": min_lift,
        "consequent_size": 1,
    },
    "metrics": {
        "itemsets": int(mining_run['itemsets']),
        "largest_itemset": int(mining_run['largest_itemset']),
        "apriori_seconds": float(mining_run['seconds']),
        "apriori_peak_memory_mb": float(mining_run['peak_memory_mb']),
    },
    "number_of_rules": int(len(rules_for_json)),
    "coverage": {
        "products_that_can_be_recommended": int(len(covered_products)),
        "total_products": int(basket.shape[1]),
        "coverage_ratio": round(len(covered_products) / basket.shape[1], 4),
    },
    "not_recommendable": not_recommendable,
    "popular_products": popular_products,
}

with open('../models/metadata.json', 'w', encoding='utf-8') as metadata_file:
    json.dump(metadata, metadata_file, ensure_ascii=False, indent=1)

print(metadata['coverage'])

{'products_that_can_be_recommended': 57, 'total_products': 98, 'coverage_ratio': 0.5816}
